# F-Smart Memories — full project trên Kaggle

Dùng source của ứng dụng đã kiểm thử local: MTCNN + AdaFace, tìm kiếm cosine, bốn preset NST và VGG19 tùy chọn.

**Chuẩn bị:** tạo notebook riêng tư, bật Internet và GPU nếu có; thêm `dist/fsmart-memories-source.zip` bằng Kaggle Dataset riêng tư. Không upload `.venv`, `runtime`, ảnh cá nhân hoặc embeddings vào dataset công khai.

Notebook này chưa được chạy trong một session Kaggle thực tế. Phần API/model đã chạy trên Windows/PyTorch CPU; file này đã qua kiểm tra cấu trúc và cú pháp.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, os, json
PROJECT = Path('/kaggle/working/fsmart-memories')
archives = list(Path('/kaggle/input').rglob('fsmart-memories-source.zip'))
if len(archives) != 1:
    raise FileNotFoundError('Add exactly one fsmart-memories-source.zip as a private Kaggle Input.')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        destination = (PROJECT / member.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()) or (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Unsafe ZIP entry')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(PROJECT)


## Cài thư viện ứng dụng

Giữ lại torch/torchvision của Kaggle. Nếu session báo lỗi ABI sau khi thay numpy, restart kernel rồi chạy lại từ đầu. Không cần gỡ CUDA hoặc cài lại torch nếu cặp thư viện hiện tại hoạt động.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt','-r','requirements-dev.txt'],check=True)
import torch, torchvision
print({'torch':torch.__version__,'torchvision':torchvision.__version__,'cuda':torch.cuda.is_available()})
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


## Tải pretrained từ upstream và kiểm tra toàn bộ ứng dụng

Cần dung lượng cho AdaFace, presets, VGG19 và ảnh mẫu. Chỉ tải lần đầu; URL/revision/checksum được lưu. Demo có ảnh công khai và query biến thể cùng ảnh, chỉ để kiểm tra chức năng; không công bố thành accuracy trên dữ liệu mới.


In [ ]:
subprocess.run([sys.executable,'scripts/fetch_assets.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
subprocess.run([sys.executable,'scripts/prepare_demo.py','--custom'],check=True)
print(json.loads(Path('artifacts/smoke/report.json').read_text()))


In [ ]:
from PIL import Image
from IPython.display import display
for style in ['candy','mosaic','rain_princess','udnie','custom_vgg19']:
    print(style)
    display(Image.open(PROJECT / 'artifacts/smoke' / f'{style}.png'))


## Tìm ảnh trên dữ liệu của bạn

Sửa hai đường dẫn. Ảnh trong gallery và query cần khác lần chụp khi đánh giá chất lượng. Nếu query có nhiều người, đặt `FACE_INDEX` theo thứ tự box được in ra.


In [ ]:
from fsmart.config import Settings
from fsmart.models import Models
from PIL import ImageOps
import numpy as np

GALLERY_DIR = Path('/kaggle/input/your-private-data/gallery')
QUERY_IMAGE = Path('/kaggle/input/your-private-data/query.jpg')
FACE_INDEX = 0
THRESHOLD = 0.4  # provisional: select on validation, then freeze for test
models = Models(Settings())

def analyze_file(path):
    with Image.open(path) as source:
        return models.analyze(ImageOps.exif_transpose(source).convert('RGB'))

def search_gallery(gallery_dir, query_path, face_index=0, threshold=0.4):
    query_faces = analyze_file(query_path)
    print('Query boxes:', [face['bbox'] for face in query_faces])
    if not query_faces or face_index >= len(query_faces):
        raise ValueError('No valid query face selected')
    q = np.asarray(query_faces[face_index]['embedding'],dtype=np.float32)
    matches=[]
    for path in sorted(gallery_dir.iterdir()):
        if path.suffix.lower() not in {'.jpg','.jpeg','.png','.webp'}: continue
        faces = analyze_file(path)
        score = max((float(np.dot(q,face['embedding'])) for face in faces),default=-1)
        if score >= threshold: matches.append((path,score))
    return sorted(matches,key=lambda row:-row[1])

if GALLERY_DIR.is_dir() and QUERY_IMAGE.is_file():
    matches=search_gallery(GALLERY_DIR,QUERY_IMAGE,FACE_INDEX,THRESHOLD)
    for path,score in matches[:20]:
        print(path.name,round(score,4))
        display(Image.open(path).resize((256,256)))
else:
    print('Set GALLERY_DIR and QUERY_IMAGE to your own private Kaggle Input paths.')


## Chuyển style một ảnh của bạn

Preset có sẵn không cần train. Với ảnh style tùy chọn, gọi `models.custom(content, reference, strength, steps, progress)`; backbone VGG19 giữ cố định và tối ưu ảnh đầu ra.


In [ ]:
CONTENT_IMAGE = Path('/kaggle/input/your-private-data/content.jpg')
STYLE = 'mosaic'
if CONTENT_IMAGE.is_file():
    with Image.open(CONTENT_IMAGE) as source:
        result=models.preset(ImageOps.exif_transpose(source).convert('RGB'),STYLE,.8,lambda _,message:print(message))
    output=PROJECT/'artifacts'/'my_stylized_image.png'
    output.parent.mkdir(exist_ok=True)
    result.save(output)
    display(result)
    print(output)
else:
    print('Set CONTENT_IMAGE to your photo.')


## Benchmark độc lập và lấy kết quả về

Tạo manifest theo `data/benchmark.example.json`, gán nhãn relevant_photo_ids rồi chạy:

```python
subprocess.run([sys.executable,'scripts/benchmark.py','--manifest','data/benchmark.json','--threshold','0.4','--k','20'],check=True)
```

Giữ ảnh validation/test độc lập; báo query không detect được mặt và query người không có trong kho. Tải `artifacts/`, `weights/provenance.json` và checkpoint cần dùng về máy. Để dùng giao diện web, chạy source local theo README; notebook không tạo public tunnel.
